# Phase 4 — fair comparison (self-contained)

Every cell is safe to re-run. Artifacts live in `/content/artifacts`, outside
the project folder, so re-running setup never deletes your 4-minute download.

**Runtime → T4 GPU → Run all.** About 20 minutes.

### 1. Setup

In [ ]:
# ---- Cell 1: setup. Safe to re-run: nothing is destroyed, nothing re-downloads. ----
import os, sys, shutil, subprocess, json, time, math, pickle
import numpy as np, statistics as st
import torch
from google.colab import drive

print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Enable the T4: Runtime -> Change runtime type -> T4 GPU")

if not os.path.ismount("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

ROOT  = "/content/drive/MyDrive"
P1    = f"{ROOT}/Phase1_Project/MemberB_B4_B6_output"
P1FIX = f"{ROOT}/Phase1_Project/data_fix_output"
GV2   = f"{ROOT}/Phase3_Project/guardrail_output_v2"
OUT   = f"{ROOT}/Phase4_Project"
os.makedirs(f"{OUT}/data", exist_ok=True)

# Artifacts live OUTSIDE the project dir so re-running setup never deletes them.
ART     = "/content/artifacts"
PROJECT = "/content/QuranicRAG"
os.makedirs(ART, exist_ok=True)
os.makedirs(f"{PROJECT}/src", exist_ok=True)
os.makedirs(f"{PROJECT}/quranNLP/shared/data", exist_ok=True)
os.chdir(PROJECT)

# Code: clone fresh, but tolerate a leftover clone from a previous run.
shutil.rmtree("/content/_repo", ignore_errors=True)
subprocess.run(["git","clone","--depth","1",
    "https://github.com/Laiba-Noor/quranic-rag-hallucination-free.git",
    "/content/_repo"], check=True)
for src in ["/content/_repo/src", f"{GV2}/src"]:       # Drive overlay wins
    for f in os.listdir(src):
        if f.endswith(".py"):
            shutil.copy(f"{src}/{f}", f"{PROJECT}/src/{f}")
print("src files:", len(os.listdir(f"{PROJECT}/src")))

CSV = f"{PROJECT}/quranNLP/shared/data/final_cross_reference_index.csv"
if not os.path.exists(CSV):
    shutil.copy(f"{P1FIX}/shared_data/final_cross_reference_index.csv", CSV)
print("corpus:", os.path.getsize(CSV)/1e6, "MB")

try:
    import hnswlib, sentence_transformers, scipy      # noqa
    print("deps already installed")
except ImportError:
    !pip install -q sentence-transformers hnswlib scipy
print("SETUP OK")

### 2. Load both systems

In [ ]:
# ---- Cell 2: pull artifacts from Drive. Skips anything already present. ----
NEEDED = {
    "m_v1": f"{P1}/b5_real_finetuned",
    "i_v1": f"{P1}/index",
    "m_v2": f"{GV2}/b5_real_finetuned_v2",
    "i_v2": f"{GV2}/index_v2",
}
t = time.time()
for local, remote in NEEDED.items():
    dst = f"{ART}/{local}"
    if os.path.isdir(dst) and os.listdir(dst):
        print(f"  have {local}")
        continue
    if not os.path.isdir(remote):
        raise SystemExit(f"MISSING IN DRIVE: {remote}")
    print(f"  copying {local} ...")
    shutil.copytree(remote, dst, dirs_exist_ok=True)
print(f"artifacts ready in {time.time()-t:.0f}s")

sys.path.insert(0, f"{PROJECT}/src")
from sentence_transformers import SentenceTransformer
from b6_build_index_and_retrieval_api import load_index, RetrievalAPI

def load_system(mdir, idir, max_seq=None):
    m = SentenceTransformer(f"{ART}/{mdir}")
    if max_seq: m.max_seq_length = max_seq
    idx, ent = load_index(dim=m.get_sentence_embedding_dimension(), out_dir=f"{ART}/{idir}")
    return RetrievalAPI(m, idx, ent), ent, m

api_v1, ent1, mod1 = load_system("m_v1", "i_v1")
api_v2, ent2, mod2 = load_system("m_v2", "i_v2")
print(f"v1 {len(ent1)} entries | v2 {len(ent2)} entries | max_seq {mod2.max_seq_length}")

### 3. Metrics

Ranks until 30 *unique verses* are found, so indexes of different sizes are compared fairly.

In [ ]:
# ---- Cell 3: fair retrieval + metrics ----
def _safe_retrieve(api, query, k):
    """hnswlib raises RuntimeError both when ef < k AND when the graph simply
    cannot produce k neighbours. Raise ef in step with k, and halve k on
    failure rather than giving up."""
    n = len(api.entries)
    k = min(k, n)
    while k >= 1:
        try:
            api.index.set_ef(min(max(k * 2, 64), n))
        except Exception:
            pass
        try:
            return api.retrieve(query, top_k=k)
        except RuntimeError:
            k //= 2
    return []

def ranked_verse_keys(api, query, want=30, cap=4096, skip_entry=None):
    """Rank until `want` unique verses are found, widening the search as needed.
    Never asks for more than `cap` or 60% of the index, both of which make
    hnswlib fail."""
    n = len(api.entries)
    hard = max(64, min(cap, int(n * 0.6)))
    k = 64
    best = []
    while True:
        kk = min(k, hard)
        hits = _safe_retrieve(api, query, kk)
        seen, out = set(), []
        for r in hits:
            if skip_entry and skip_entry(r):
                continue
            vk = r["verse_key"]
            if vk not in seen:
                seen.add(vk); out.append(vk)
                if len(out) >= want:
                    return out
        if len(out) > len(best):
            best = out
        if kk >= hard or not hits:
            return best
        k *= 4

def evaluate(api, gold_pairs, ks=(1, 5, 10, 20), skip_entry=None, want=30):
    from collections import defaultdict
    acc = defaultdict(list)
    per_q = {"rr": [], "r10": [], "hit20": []}
    for q, gold in gold_pairs:
        ranked = ranked_verse_keys(api, q, want=want, skip_entry=skip_entry)
        rr = next((1.0 / i for i, vk in enumerate(ranked, 1) if vk in gold), 0.0)
        acc["MRR"].append(rr)
        dcg = sum(1 / math.log2(i + 1) for i, vk in enumerate(ranked[:10], 1) if vk in gold)
        idcg = sum(1 / math.log2(i + 1) for i in range(1, min(len(gold), 10) + 1))
        acc["NDCG@10"].append(dcg / idcg if idcg else 0.0)
        for k in ks:
            h = sum(1 for vk in ranked[:k] if vk in gold)
            acc[f"Recall@{k}"].append(h / len(gold))
            acc[f"HitRate@{k}"].append(1.0 if h else 0.0)
        per_q["rr"].append(rr)
        per_q["r10"].append(sum(1 for vk in ranked[:10] if vk in gold) / len(gold))
        per_q["hit20"].append(1.0 if any(vk in gold for vk in ranked[:20]) else 0.0)
    return {m: sum(v) / len(v) for m, v in acc.items()}, per_q

def show(title, rows):
    print(f"\n{title}")
    print(f"{'system':<28}{'R@10':>8}{'R@20':>8}{'Hit@10':>9}{'Hit@20':>9}{'MRR':>8}{'NDCG':>8}")
    print("-" * 78)
    for name, r in rows:
        print(f"{name:<28}{r['Recall@10']:>8.4f}{r['Recall@20']:>8.4f}"
              f"{r['HitRate@10']:>9.4f}{r['HitRate@20']:>9.4f}"
              f"{r['MRR']:>8.4f}{r['NDCG@10']:>8.4f}")

def compare(label, pq_a, pq_b, keys=("r10", "hit20")):
    from scipy.stats import wilcoxon
    for kk in keys:
        a, b = pq_a[kk], pq_b[kk]
        d = [y - x for x, y in zip(a, b)]
        if not any(d):
            print(f"{label} {kk:<7} identical"); continue
        _, p = wilcoxon(a, b, zero_method="wilcox")
        sd = st.pstdev(d) or 1e-9
        print(f"{label} {kk:<7} delta={st.mean(d):+.4f}  p={p:.4g}  "
              f"d={st.mean(d)/sd:+.3f}  {'SIGNIFICANT' if p < 0.05 else 'ns'}")

# ground truth (stashed in Drive after first upload)
STASH = f"{OUT}/data/ayatec_records.json"
if not os.path.exists(STASH):
    for c in [f"{ROOT}/Phase2_Project/Roma_output/data/ayatec_records.json",
              "/content/_repo/Data/ayatec_records.json"]:
        if os.path.exists(c):
            shutil.copy(c, STASH); break
if not os.path.exists(STASH):
    from google.colab import files
    print("Upload ayatec_records.json from Research/Colab_Cells/")
    up = files.upload()
    shutil.copy(list(up.keys())[0], STASH)

aya = json.load(open(STASH, encoding="utf-8"))
GOLD = [(r["question"], set(r["verse_keys"])) for r in aya
        if r.get("question") and r.get("verse_keys")]
ceil10 = sum(min(10, len(g)) / len(g) for _, g in GOLD) / len(GOLD)
print(f"{len(GOLD)} questions | max achievable Recall@10 = {ceil10:.4f}")

### 4. Ablation — verses vs tafsir (free, no re-indexing)

In [ ]:
# ---- Cell 4: which half of the index carries the signal? (no re-indexing) ----
only_verse  = lambda r: r["source_type"] != "verse"
only_tafsir = lambda r: r["source_type"] != "tafsir"

results, perq = {}, {}
for name, api in [("v1", api_v1), ("v2", api_v2)]:
    for tag, skip in [("all entries", None), ("VERSES only", only_verse),
                      ("tafsir only", only_tafsir)]:
        key = f"{name} - {tag}"
        results[key], perq[key] = evaluate(api, GOLD, skip_entry=skip)
        print("done:", key)

show("ablation: verses vs tafsir", list(results.items()))
print()
compare("v2 verses-only vs all ", perq["v2 - all entries"], perq["v2 - VERSES only"])
compare("v2 vs v1 (all entries) ", perq["v1 - all entries"], perq["v2 - all entries"])

### 5. Matched-window chunking

250-char chunks that fit the 64-token window the model was trained with.

In [ ]:
# ---- Cell 5: matched-window chunking (250 chars, model window left at 64) ----
import csv, hnswlib
csv.field_size_limit(sys.maxsize)
WIN, STRIDE = 250, 200

def make_chunks(t):
    if len(t) <= WIN:
        return [t]
    out = [t[i:i+WIN] for i in range(0, len(t), STRIDE)]
    return [c for c in out if c.strip()]

rows_csv = list(csv.DictReader(open(CSV, encoding="utf-8")))
passage_chunks, all_chunks, cix = {}, [], {}
for r in rows_csv:
    p = (r.get("tafsir_passage") or "").strip()
    if not p or str(r.get("has_direct_tafsir")) != "True" or p in passage_chunks:
        continue
    ids = []
    for c in make_chunks(p):
        if c not in cix:
            cix[c] = len(all_chunks); all_chunks.append(c)
        ids.append(cix[c])
    passage_chunks[p] = ids

print(f"unique passages {len(passage_chunks)} | unique chunks {len(all_chunks):,} | "
      f"avg {sum(len(v) for v in passage_chunks.values())/len(passage_chunks):.1f}/passage")

m = SentenceTransformer(f"{ART}/m_v2")
m.max_seq_length = 64                      # matched to how it was fine-tuned
t = time.time()
cemb = m.encode(all_chunks, convert_to_numpy=True, normalize_embeddings=True,
                show_progress_bar=True, batch_size=256).astype(np.float32)
verses = [(r["verse_key"], (r.get("clean_verse") or "").strip())
          for r in rows_csv if (r.get("clean_verse") or "").strip()]
vemb = m.encode([v for _, v in verses], convert_to_numpy=True, normalize_embeddings=True,
                show_progress_bar=True, batch_size=256).astype(np.float32)
print(f"encoded in {(time.time()-t)/60:.1f} min")

### 6. Build chunked index

In [ ]:
# ---- Cell 6: build the chunked index ----
entries, rowsel = [], []
for i, (vk, vt) in enumerate(verses):
    entries.append({"text": vt, "verse_key": vk, "source_type": "verse"})
    rowsel.append(("v", i))
for r in rows_csv:
    p = (r.get("tafsir_passage") or "").strip()
    if not p or str(r.get("has_direct_tafsir")) != "True":
        continue
    for cid in passage_chunks[p]:
        entries.append({"text": all_chunks[cid], "verse_key": r["verse_key"],
                        "source_type": "tafsir"})
        rowsel.append(("c", cid))

vecs = np.empty((len(entries), cemb.shape[1]), dtype=np.float32)
for j, (kind, i) in enumerate(rowsel):
    vecs[j] = vemb[i] if kind == "v" else cemb[i]
print(f"entries {len(entries):,} | vectors {vecs.nbytes/1e9:.2f} GB")

ix = hnswlib.Index(space="cosine", dim=vecs.shape[1])
ix.init_index(max_elements=len(entries), ef_construction=200, M=16)
ix.add_items(vecs, ids=np.arange(len(entries)))
ix.set_ef(128)
api_ch = RetrievalAPI(m, ix, entries)
print("chunked index built")

### 7. Results

In [ ]:
# ---- Cell 7: final table ----
r_ch,  pq_ch  = evaluate(api_ch, GOLD)
r_chv, pq_chv = evaluate(api_ch, GOLD, skip_entry=only_verse)

show("FINAL COMPARISON", [
    ("v1 baseline",              results["v1 - all entries"]),
    ("v2 baseline",              results["v2 - all entries"]),
    ("v2 verses-only",           results["v2 - VERSES only"]),
    ("v2 chunked 250/64",        r_ch),
    ("v2 chunked 250/64, verses", r_chv),
])
print(f"\n(max achievable Recall@10 = {ceil10:.4f})\n")
compare("chunked vs v2        ", perq["v2 - all entries"], pq_ch)
compare("chunked-verses vs v2 ", perq["v2 - all entries"], pq_chv)

payload = {"n_questions": len(GOLD), "ceiling_recall10": ceil10,
           "config": {"window": WIN, "stride": STRIDE, "max_seq_length": 64,
                      "n_entries": len(entries)},
           **{k: v for k, v in results.items()},
           "v2 chunked": r_ch, "v2 chunked verses-only": r_chv}
json.dump(payload, open(f"{OUT}/phase4_fair_comparison.json", "w"),
          indent=2, ensure_ascii=False)
print("\nsaved:", f"{OUT}/phase4_fair_comparison.json")